# 03 — Segmentation RFM & KMeans

## Ce que je vais faire ici
je pars de notre table de 4 518 clients et on les classe en **groupes** 
selon leur comportement. À la fin, chaque client aura :
- Un **segment marketing** (Champion, À risque, Perdu, etc.) → lisible business
- Un **cluster KMeans** (numéro de groupe) → découvert par l'algorithme

## Deux approches, deux usages
- **Partie A — RFM classique** : la méthode marketing standard, basée sur 
  des règles claires. Idéale pour communiquer avec une équipe CRM.
- **Partie B — KMeans** : l'approche data-driven, qui laisse les données 
  parler. Idéale pour découvrir des profils inattendus.

je compares les deux à la fin.

## Important : on n'utilise PAS la cible CHURN ici
La segmentation est **descriptive**, pas prédictive. je veut comprendre 
"qui sont mes clients", pas "qui va churner". La prédiction, ce sera 
l'étape 4.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# Chargement
DATA_PROCESSED = Path("../data/processed")
df = pd.read_csv(DATA_PROCESSED / "customer_features.csv")

print(f"✅ Table chargée : {len(df):,} clients × {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")

✅ Table chargée : 4,518 clients × 13 colonnes

Colonnes : ['customer_id', 'churn', 'n_orders_future', 'recency', 'frequency', 'monetary', 'avg_basket', 'std_basket', 'n_distinct_products', 'tenure_days', 'avg_interpurchase_days', 'n_cancellations', 'cancellation_rate']


# PARTIE A — SEGMENTATION RFM RULES-BASED

## 2. Calcul des scores R, F, M (1 à 5)

je découpe chaque dimension RFM en **5 groupes égaux** (quintiles) :
- Pour la **Recency** : plus c'est récent, mieux c'est → un client avec 
  une faible Recency obtient le score 5
- Pour la **Frequency** : plus c'est élevé, mieux c'est → score 5 pour 
  les clients qui commandent le plus
- Pour le **Monetary** : idem, plus c'est élevé, mieux c'est → score 5 
  pour les gros dépensiers

Chaque client obtient donc 3 scores entre 1 et 5.


In [2]:
# Score Recency : inversé (moins de jours = mieux = score 5)
df['R_score'] = pd.qcut(df['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Score Frequency : direct (plus = mieux), avec gestion des doublons
df['F_score'] = pd.qcut(df['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Score Monetary : direct
df['M_score'] = pd.qcut(df['monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

print("=== Distribution des scores ===\n")
print("R_score :"); print(df['R_score'].value_counts().sort_index())
print("\nF_score :"); print(df['F_score'].value_counts().sort_index())
print("\nM_score :"); print(df['M_score'].value_counts().sort_index())

=== Distribution des scores ===

R_score :
R_score
1    904
2    902
3    900
4    885
5    927
Name: count, dtype: int64

F_score :
F_score
1    904
2    903
3    904
4    903
5    904
Name: count, dtype: int64

M_score :
M_score
1    904
2    903
3    904
4    903
5    904
Name: count, dtype: int64


## 3. Score RFM combiné

je colle les 3 scores pour obtenir un code à 3 chiffres :
- "555" = client parfait (récent, fréquent, gros dépensier)
- "111" = client perdu (ancien, peu fréquent, faible dépense)
- "511" = nouveau client qui n'a pas encore beaucoup acheté

Ce code à 3 chiffres va nous servir à attribuer le **segment métier** 
à la cellule suivante.

In [5]:
df['RFM_score'] = df['R_score'].astype(str) + df['F_score'].astype(str) + df['M_score'].astype(str)

print("Top 10 des codes RFM les plus fréquents :")
print(df['RFM_score'].value_counts().head(10))

Top 10 des codes RFM les plus fréquents :
RFM_score
555    403
111    234
455    186
121    142
344    134
444    129
211    123
122    113
544    111
233    102
Name: count, dtype: int64


## 4. Attribution des segments métier

je traduit les codes RFM en **segments compréhensibles**. 
On utilise les 11 segments standards du marketing :

| Segment | Profil |
|---|---|
| Champions | Mes meilleurs clients, à fidéliser |
| fidèle | Fidèles, ambassadeurs potentiels |
| potentiellement fidèle | À fidéliser activement |
| Nouveaux Clients | Nouveaux, à convertir |
| Prometteur | Bon début, à accompagner |
| Besoin d'attention | Moyens, à surveiller |
| Sur le point de partir | Activité en baisse |
| En Danger | Bons clients en train de partir → URGENT |
| je ne veux pas les perdre | Anciens VIPs absents → URGENCE MAX |
| en hibernation | Quasi inactifs |
| Perdu | Probablement perdus |


In [8]:
def assign_segment(row):
    r, f = row['R_score'], row['F_score']
    
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 4:
        return 'Fidèles'
    elif r >= 4 and f >= 2:
        return 'Potentiellement Fidèles'
    elif r == 5 and f == 1:
        return 'Nouveaux Clients'
    elif r >= 3 and f == 1:
        return 'Prometteurs'
    elif r == 3 and f >= 2 and f <= 3:
        return 'Besoin d\'Attention'
    elif r == 2 and f >= 2 and f <= 3:
        return 'Sur le point de Partir'
    elif r <= 2 and f >= 3:
        return 'En danger'
    elif r == 1 and f >= 4:
        return "Je ne veux pas les perdre"
    elif r == 2 and f == 1:
        return 'en Hibernation'
    else:
        return 'Perdus'

df['segment_rfm'] = df.apply(assign_segment, axis=1)

# Distribution
segment_dist = df['segment_rfm'].value_counts()
segment_pct = (df['segment_rfm'].value_counts(normalize=True) * 100).round(1)

print("Distribution des segments RFM :")
print(pd.DataFrame({'n_clients': segment_dist, 'pct': segment_pct}))

Distribution des segments RFM :
                         n_clients   pct
segment_rfm                             
Champions                     1161  25.7
Perdus                         670  14.8
Potentiellement Fidèles        495  11.0
Sur le point de Partir         471  10.4
En danger                      422   9.3
Fidèles                        390   8.6
Besoin d'Attention             373   8.3
en Hibernation                 243   5.4
Prometteurs                    239   5.3
Nouveaux Clients                54   1.2
